## 1 · Imports

Lädt alle externen Bibliotheken, die im Notebook verwendet werden.

| Gruppe | Bibliothek | Zweck |
|---|---|---|
| Standardbibliothek | `pathlib.Path` | Plattformunabhängige Dateipfade |
| Standardbibliothek | `time`, `logging` | Wartezeiten zwischen API-Aufrufen, strukturierte Logs |
| Datenverarbeitung | `pandas` | Tabellen (DataFrames), Parquet lesen/schreiben (Parquet-Backend `pyarrow` muss installiert, aber nicht importiert sein) |
| HTTP & API | `requests` | HTTP-Anfragen an die öffentliche Binance REST API (kein Auth nötig) |

In [1]:
# ── Standardbibliotheken ──────────────────────────────────────
from pathlib import Path
import sys
import time
import logging

# ── Datenverarbeitung ─────────────────────────────────────────
import pandas as pd

# ── HTTP & API ────────────────────────────────────────────────
import requests

## 2 · Logging-Konfiguration

Richtet das Python-Logging ein, das alle Statusmeldungen während der Datenbeschaffung ausgibt.

Das Format `line: %(lineno)-4s %(levelname)-8s %(message)s` zeigt bei jeder Meldung:
- die **Zeilennummer** im Code (hilfreich zur Fehlersuche)
- das **Level** (`INFO` = normale Information, `WARNING` = unvollständige Daten oder übersprungene Symbole)
- die eigentliche **Nachricht**

`force=True` stellt sicher, dass die Konfiguration auch bei erneutem Ausführen der Zelle überschrieben wird. Die Variable `log` wird im gesamten Notebook als einheitliche Schnittstelle für alle Ausgaben verwendet.

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="line: %(lineno)-4s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger(__name__)


## 3 · Pfade & Konfiguration

Alle Projektverzeichnisse (`PROJECT_ROOT`, `DATA_DIR`, `RAW_DIR`, `FETCHED_DIR`,
`PROCESSED_DIR`) sowie `ASSETS_PATH` und `OHLCV_DIR` kommen aus `notebooks/config.py`
(gemeinsam mit `01_assets.ipynb`/`01_baskets.ipynb`), damit das Notebook auf jedem Rechner
ohne Anpassung funktioniert und alle drei Notebooks garantiert auf dieselben Pfade zeigen.

**Wichtige Pfade:**
- `ASSETS_PATH`  Fixe Asset-Liste (aus `01_assets.ipynb`, inkl. `binance_symbol`-Spalte)
- `ASSET_OHLCV_5M_DIR`  Zielordner für die OHLCV-Parquet-Dateien (lokaler Alias für `OHLCV_DIR`)

**API-Parameter (notebook-spezifisch, bleiben lokal):**
- `INTERVAL = "5m"`  Jede Kerze deckt 5 Minuten ab → 288 Kerzen pro Tag
- `BINANCE_KLINE_LIMIT = 1000`  Maximale Kerzen pro API-Anfrage (Binance-Limit)
- `SLEEP_BETWEEN_REQUESTS`  Pause zwischen Anfragen, um Rate-Limits zu vermeiden
- `OHLCV_START_DATE` / `OHLCV_END_DATE`  Eigenständiges 2-Jahres-Fenster für die OHLCV-Beschaffung, unabhängig von `asset_daily`'s tagesvariabler Eligibility — die OHLCV-Universe ist jetzt eine fixe Asset-Liste über den ganzen Zeitraum.

`controlpath()` prüft beim Start, ob alle erwarteten Dateien und Ordner vorhanden sind, und gibt deren Inhalt aus.

In [3]:
# ── Shared paths — from config.py (also used by 01_assets.ipynb / 01_baskets.ipynb) ──
sys.path.insert(0, str(Path("../..").resolve()))
from src.config import (
    PROJECT_ROOT, DATA_DIR, RAW_DIR, FETCHED_DIR, PROCESSED_DIR,
    ASSETS_PATH, OHLCV_DIR,
)

ASSET_OHLCV_5M_DIR = OHLCV_DIR  # local alias, keeps the name used throughout this notebook

# Fetch window. Matches 01_assets.ipynb's configured START_DATE/END_DATE exactly.
# END_DATE is kept 2 days behind "today" as a safety margin against UTC/local-timezone
# edge cases and partially-elapsed days (Binance itself does not reject or flag
# incomplete-day requests - it just silently returns fewer candles - so this margin
# avoids ever asking for a day that might not be fully closed yet in UTC).
# Already-complete months (e.g. 2026-06) are skipped automatically on re-run, and the
# window can be safely extended forward later at zero cost once more days have passed.
OHLCV_START_DATE = "2024-07-23"
OHLCV_END_DATE = "2026-07-22"  # inclusive

INTERVAL               = "5m"
BINANCE_KLINE_LIMIT = 1000

def controlpath():
    log.info(f"Projektverzeichnis:            {PROJECT_ROOT}")
    log.info(f"Datenverzeichnis:              {DATA_DIR}")
    log.info(f"Rohdatenverzeichnis:           {RAW_DIR}")
    log.info(f"Gefetchte Daten:               {FETCHED_DIR}")
    log.info(f"Verarbeitete Daten:            {PROCESSED_DIR}")
    print()

    checks = {
        "Assets-Datei":              ASSETS_PATH,
        "Asset-OHLCV-5M-Verzeichnis": ASSET_OHLCV_5M_DIR,
    }

    for name, path in checks.items():
        exists = path.exists()
        symbol = "✅" if exists else "❌ FEHLT"
        log.info(f"  {symbol}  {name}: {path}")

        # Falls Ordner → Inhalt anzeigen
        if exists and path.is_dir():
            files = sorted(path.glob("*.parquet"))
            if files:
                for f in files:
                    log.info(f"       📄 {f.name}")
            else:
                log.info("       ⚠️  Ordner ist leer — keine .parquet Dateien!")


controlpath()

line: 24    INFO      Projektverzeichnis:            C:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit
line: 25    INFO      Datenverzeichnis:              C:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data
line: 26    INFO      Rohdatenverzeichnis:           C:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw
line: 27    INFO      Gefetchte Daten:               C:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched
line: 28    INFO      Verarbeitete Daten:            C:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\processed
line: 39    INFO        ✅  Assets-Datei: C:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\assets.parquet
line: 39    INFO        ✅  Asset-OHLCV-5M-Verzeichnis: C:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\asset_ohlcv_5m
line: 46    INFO             📄 ohlcv_2024-07.parquet
line: 46    INFO             📄 ohlc

## 4 · OHLCV-Datenbeschaffung

Dieses Block enthält die gesamte Logik zum Herunterladen der Kerzendaten von Binance.

### `load_valid_binance_symbols()`
Fragt einmalig die Binance-API nach allen aktiven Handelssymbolen (`/api/v3/exchangeInfo`). Dient hier nur noch als defensive Zweitprüfung — die eigentliche Symbol-Auflösung passiert bereits robust in `01_assets.ipynb` über CoinGecko's Ticker-Daten.

### `load_fixed_asset_universe()`
Liest `assets.parquet` und filtert auf Assets mit aufgelöstem `binance_symbol`. Anders als früher ist diese Liste **fix für den ganzen Zeitraum** — es gibt kein tägliches Dictionary mehr, weil die Asset-Auswahl nicht mehr täglich neu bestimmt wird.

### `json_to_parquet()`
Speichert die heruntergeladenen Rohdaten als Parquet-Datei. Dabei werden die Daten nach Jahr und Monat aufgeteilt (`ohlcv_2026-01.parquet`, `ohlcv_2026-02.parquet`, …). Bereits vorhandene Einträge werden mit `drop_duplicates` nicht doppelt gespeichert. Die `timestamp`-Spalte wird beim Schreiben (und beim Einlesen bestehender Dateien vor dem Merge) explizit auf `TIMESTAMP_DTYPE` normalisiert — pandas ≥2.0 kann sonst je nach Umgebung/Version unterschiedliche `datetime64`-Auflösungen (`ns` vs. `ms`) über verschiedene Monatsdateien hinweg erzeugen, was Abschnitt 6's Schema-Check zurecht als Fehler meldet.

### `fetch_ohlcv_for_range()`  Hauptfunktion
Steuert den gesamten Ablauf:
1. Lädt die fixe Asset-Universe und gültige Binance-Symbole
2. Iteriert pro **Monat** (nicht pro Tag) über `OHLCV_START_DATE`…`OHLCV_END_DATE`
3. Prüft für jeden Monat welche Tage bereits vollständig gespeichert sind
    Ein Tag gilt als vollständig wenn der **Median** aller Ticker ≥ 270 Kerzen hat
4. Fetcht nur die fehlenden Tage/Ticker von Binance
5. Speichert die Ergebnisse via `json_to_parquet()`

**Steuerungsvariable:**

| `FORCE_REFETCH` | Verhalten |
|---|---|
| `False` | Normalbetrieb  vollständige Monate werden sofort übersprungen |
| `True` | Reparatur-Modus  jeden Ticker pro Tag einzeln prüfen und nur fehlende nachfetchen |

Quelle: `docs/uml/ohlcv_UML.drawio` (mit draw.io/diagrams.net öffnen, PNG-Export für die
Anzeige unten aktualisieren).

![UML](../../docs/uml/ohlcv_UML.png)

In [4]:
# =========================
# Konfiguration
# =========================
BASE_URL               = "https://data-api.binance.vision"
SLEEP_BETWEEN_REQUESTS = 0.05

# False  →  nur fehlende/unvollständige Tage fetchen (Normalbetrieb)
# True   →  jeden Ticker pro Tag prüfen und fehlende nachfetchen (Reparatur-Modus)
FORCE_REFETCH = False

# datetime64 resolution all parquet files are normalized to on write. Pinning this
# explicitly avoids schema drift across pandas/pyarrow versions or machines (pandas >=2.0
# preserves the unit given to pd.to_datetime(unit=...) instead of always upcasting to ns,
# so files written by different environments can otherwise end up with different units).
TIMESTAMP_DTYPE = "datetime64[ms, UTC]"

# =========================
# Public REST
# =========================
SESSION = requests.Session()

def load_valid_binance_symbols() -> set:
    """Einmalig alle gültigen Spot-Symbole von Binance laden. Dient hier nur noch als
    defensive Prüfung (z.B. seit der Symbol-Auflösung delistete Paare) — die
    Symbol-Auflösung selbst kommt bereits robust aus assets.parquet (binance_symbol)."""
    info = SESSION.get(f"{BASE_URL}/api/v3/exchangeInfo", timeout=20).json()
    return {s["symbol"] for s in info["symbols"] if s["status"] == "TRADING"}

VALID_BINANCE_SYMBOLS: set = set()   # wird in fetch_ohlcv_for_range befüllt

def public_get(path: str, params: dict | None = None):
    url = f"{BASE_URL}{path}"
    r = SESSION.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Binance Fehler {r.status_code}: {r.text}")
    return r.json()


# =========================
# Parquet Speicherung
# =========================
def json_to_parquet(data: list[dict]) -> None:
    ASSET_OHLCV_5M_DIR.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(data)
    df["timestamp"] = df["timestamp"].dt.tz_convert("UTC").astype(TIMESTAMP_DTYPE)
    df["_year"]     = df["timestamp"].dt.year
    df["_month"]    = df["timestamp"].dt.month

    for (year, month), month_df in df.groupby(["_year", "_month"]):
        month_df = month_df.drop(columns=["_year", "_month"])
        out      = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        if out.exists():
            existing_df = pd.read_parquet(out)
            existing_df["timestamp"] = existing_df["timestamp"].astype(TIMESTAMP_DTYPE)
            df_all = (
                pd.concat([existing_df, month_df], ignore_index=True)
                .drop_duplicates(subset=["timestamp", "market_symbol"])
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            log.info(f"  [PARQUET] Aktualisiert: {out.name}  →  {len(df_all):,} Zeilen")
        else:
            df_all = (
                month_df
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            log.info(f"  [PARQUET] Erstellt: {out.name}  →  {len(df_all):,} Zeilen")

        df_all.to_parquet(out, index=False)


# =========================
# Fixe Asset-Universe laden
# =========================
def load_fixed_asset_universe() -> pd.DataFrame:
    """Liest nur assets.parquet (nicht mehr asset_daily) und liefert eine flache,
    fixe Liste — die OHLCV-Universe ändert sich nicht mehr täglich, sondern jedes
    Asset bekommt die volle Historie über den ganzen Zeitraum."""
    log.info("─" * 60)
    log.info(" Fixe Asset-Universe laden")
    log.info("─" * 60)
    df_assets = pd.read_parquet(ASSETS_PATH)
    log.info(f"  → {len(df_assets)} Assets geladen")

    fixed_df = df_assets.loc[df_assets["binance_symbol"].notna()].copy()
    fixed_df = fixed_df.rename(columns={"binance_symbol": "market_symbol"})
    log.info(f"  → {len(fixed_df)} Assets mit aufgelöstem Binance-Symbol")

    dropped = df_assets.loc[df_assets["binance_symbol"].isna(), ["coingecko_id", "symbol", "name"]]
    if not dropped.empty:
        log.warning(f"  ⚠️  {len(dropped)} Assets ohne Binance-Symbol — werden übersprungen: {dropped['symbol'].tolist()}")

    return fixed_df[["asset_id", "coingecko_id", "symbol", "name", "market_symbol"]].reset_index(drop=True)


# =========================
# Hauptfunktion
# =========================
def fetch_ohlcv_for_range() -> None:
    mode = "REPARATUR" if FORCE_REFETCH else "NORMAL"
    log.info("=" * 60)
    log.info(f"  OHLCV-DATENBESCHAFFUNG GESTARTET  [{mode}]")
    log.info("=" * 60)

    fixed_universe_df = load_fixed_asset_universe()

    global VALID_BINANCE_SYMBOLS
    VALID_BINANCE_SYMBOLS = load_valid_binance_symbols()
    log.info(f"  → {len(VALID_BINANCE_SYMBOLS)} gültige Binance-Symbole geladen")

    all_symbols = fixed_universe_df["market_symbol"].tolist()
    invalid     = [s for s in all_symbols if s not in VALID_BINANCE_SYMBOLS]
    symbols     = [s for s in all_symbols if s in VALID_BINANCE_SYMBOLS]
    if invalid:
        log.warning(f"  ⚠️  {len(invalid)} Symbole aktuell nicht auf Binance gelistet — werden übersprungen: {invalid}")

    window_start = pd.Timestamp(OHLCV_START_DATE).date()
    window_end   = pd.Timestamp(OHLCV_END_DATE).date()
    month_starts = pd.date_range(
        pd.Timestamp(window_start).replace(day=1), window_end, freq="MS"
    )

    for month_start_ts in month_starts:
        year, month   = month_start_ts.year, month_start_ts.month
        month_first   = pd.Timestamp(year=year, month=month, day=1).date()
        days_in_month = month_start_ts.days_in_month
        month_last    = pd.Timestamp(year=year, month=month, day=days_in_month).date()

        all_dates = pd.date_range(max(month_first, window_start), min(month_last, window_end), freq="D").date

        out = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        # Vorhandene Daten einlesen: Kerzen pro (Tag, Ticker)
        existing_dates  = set()
        ticker_candles  = {}   # (date, market_symbol) → Kerzenanzahl
        if out.exists():
            df_ex         = pd.read_parquet(out, columns=["timestamp", "market_symbol"])
            df_ex["date"] = pd.to_datetime(df_ex["timestamp"]).dt.date
            per_ticker    = df_ex.groupby(["date", "market_symbol"]).size()
            ticker_candles = per_ticker.to_dict()
            day_median    = per_ticker.groupby("date").median()
            existing_dates = set(day_median[day_median >= 270].index)
            incomplete     = set(day_median[day_median < 270].index)
            if incomplete:
                log.warning(f"  ⚠️  Unvollständige Tage: {sorted(incomplete)}")

        missing_dates = [d for d in all_dates if d not in existing_dates]

        log.info(f"\n{'=' * 60}")
        log.info(f"  Monat {year}-{month:02d}:  {len(all_dates)} Tage im Fenster  |  {len(symbols)} Ticker")
        log.info(f"  Vorhanden ({len(existing_dates)}) / Fehlend ({len(missing_dates)}): {missing_dates}")
        log.info(f"{'=' * 60}")

        if not FORCE_REFETCH:
            # Normalbetrieb: nur komplett fehlende Tage fetchen
            if not missing_dates:
                log.info("  ✅ Alle Tage vollständig — Monat wird übersprungen")
                continue
            dates_to_process = missing_dates
        else:
            # Reparatur-Modus: alle Tage im Fenster durchgehen, pro Ticker prüfen
            log.info(f"  ⚠️  REPARATUR-MODUS: alle {len(all_dates)} Tage im Fenster werden geprüft")
            dates_to_process = list(all_dates)

        fetch_errors = []

        for current_date in dates_to_process:
            # Welche Ticker brauchen noch Daten für diesen Tag?
            if FORCE_REFETCH:
                symbols_to_fetch = [
                    s for s in symbols
                    if ticker_candles.get((current_date, s), 0) < 270
                ]
                if not symbols_to_fetch:
                    log.info(f"  [✔] {current_date} — alle Ticker vollständig")
                    continue
                log.info(f"\n  ── {current_date}: {len(symbols_to_fetch)}/{len(symbols)} Ticker fehlen ──")
            else:
                symbols_to_fetch = symbols
                log.info(f"\n  ── Fetche {current_date} ({len(symbols)} Ticker) ──")

            all_rows = []
            day_errors = []

            for market_symbol in symbols_to_fetch:
                try:
                    raw = public_get("/api/v3/klines", params={
                        "symbol":    market_symbol,
                        "interval":  INTERVAL,
                        "startTime": int(pd.Timestamp(current_date, tz="UTC").timestamp() * 1000),
                        "endTime":   int((pd.Timestamp(current_date, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000) - 1,
                        "limit":     BINANCE_KLINE_LIMIT,
                    })
                    for row in raw:
                        all_rows.append({
                            "timestamp":     pd.to_datetime(int(row[0]), unit="ms", utc=True),
                            "market_symbol": market_symbol,
                            "open":          float(row[1]),
                            "high":          float(row[2]),
                            "low":           float(row[3]),
                            "close":         float(row[4]),
                            "volume_base":   float(row[5]),
                            "volume_quote":  float(row[7]),
                            "trade_count":   int(row[8]),
                        })
                    time.sleep(SLEEP_BETWEEN_REQUESTS)
                except Exception as e:
                    msg = f"{current_date} | {market_symbol}: {e}"
                    day_errors.append(msg)
                    fetch_errors.append(msg)

            log.info(f"  → {len(all_rows):,} Kerzen  |  {len(day_errors)} Fehler")
            for err in day_errors:
                log.warning(f"    [FEHLER] {err}")

            if all_rows:
                json_to_parquet(all_rows)
            else:
                log.info(f"  [SKIP] Keine Daten für {current_date}")

        if FORCE_REFETCH and fetch_errors:
            log.warning(f"\n  ❌ Reparatur {year}-{month:02d}: {len(fetch_errors)} Fehler gesamt:")
            for err in fetch_errors:
                log.warning(f"    {err}")

    log.info("\n" + "=" * 60)
    log.info("  OHLCV-DATENBESCHAFFUNG ABGESCHLOSSEN")
    log.info("=" * 60)


# =========================
# Ausführen
# =========================
fetch_ohlcv_for_range()


line: 103   INFO      ============================================================
line: 104   INFO        OHLCV-DATENBESCHAFFUNG GESTARTET  [NORMAL]
line: 105   INFO      ============================================================
line: 81    INFO      ────────────────────────────────────────────────────────────
line: 82    INFO       Fixe Asset-Universe laden
line: 83    INFO      ────────────────────────────────────────────────────────────
line: 85    INFO        → 57 Assets geladen
line: 89    INFO        → 47 Assets mit aufgelöstem Binance-Symbol
line: 93    WARNING     ⚠️  10 Assets ohne Binance-Symbol — werden übersprungen: ['CC', 'CRO', 'HYPE', 'KAS', 'LIT', 'MNT', 'STABLE', 'VVV', 'XDC', 'FLR']
line: 111   INFO        → 1354 gültige Binance-Symbole geladen
line: 151   INFO      
line: 152   INFO        Monat 2024-07:  9 Tage im Fenster  |  47 Ticker
line: 153   INFO        Vorhanden (9) / Fehlend (0): []
line: 154   INFO      ==================================================

## 5 · Qualitätsprüfung  `ohlcvCheck()`

Liest alle gespeicherten OHLCV-Parquet-Dateien und prüft sie **gegen die Soll-Werte** —
nicht nur, was zufällig in den Dateien steht:

- **Soll-Ticker**: die fixe Asset-Universe aus `load_fixed_asset_universe()` (gefiltert auf aktuell gültige Binance-Symbole)
- **Soll-Tage**: jeder Kalendertag zwischen `OHLCV_START_DATE` und `OHLCV_END_DATE`, der in den jeweiligen Monat fällt

Für jede (Tag, Ticker)-Kombination wird die Kerzenanzahl einzeln bewertet — ein
Durchschnitt über alle Ticker würde einzelne lückenhafte Ticker verstecken, und Ticker
oder Tage, die komplett fehlen (0 Zeilen), würden beim reinen Auslesen der Datei gar
nicht erst auftauchen.

**Bewertung pro (Tag, Ticker):**
- ✅ ≥ 274 Kerzen  vollständig (288 = 24 h × 12 Intervalle à 5 min)
- ⚠️ 200–273 Kerzen  teilweise vorhanden
- ❌ < 200 Kerzen  unvollständig

**Listing-bewusste Behandlung:**
- Pro Ticker wird zuerst das **First-Seen-Datum** ermittelt (erster Tag mit ≥ 1 Kerze,
  über alle Monatsdateien hinweg).
- Tage **vor** diesem Datum zählen nicht als Fehler und tauchen in der Tages- und
  Lücken-Statistik nicht auf — der Coin war zu diesem Zeitpunkt schlicht noch nicht
  gelistet.
- Stattdessen wird der Ticker **genau einmal**, am Tag seines ersten Auftretens,
  als "🆕 Neu gelistet" gemeldet.
- Erst **ab dem Tag nach** dem First-Seen-Datum zählen fehlende/unvollständige Tage
  als echte Lücke — der Listing-Tag selbst kann strukturell nie 288 Kerzen haben
  (Listing passiert meist mitten am Tag, nicht um 00:00 UTC).

Diese Zelle hat **keinen Einfluss** auf die gespeicherten Daten  sie dient nur zur
Kontrolle.

In [5]:
import datetime

# =========================
# OHLCV Schnellübersicht — gegen Soll-Ticker & Soll-Zeitraum, listing-bewusst
# =========================
def ohlcvCheck():
    log.info("=" * 70)
    log.info("OHLCV DATEI-ÜBERSICHT")
    log.info("=" * 70)

    # Soll-Ticker: fixe Asset-Universe, gefiltert auf aktuell gültige Binance-Symbole
    fixed_universe_df = load_fixed_asset_universe()
    global VALID_BINANCE_SYMBOLS
    valid_symbols = VALID_BINANCE_SYMBOLS or load_valid_binance_symbols()
    VALID_BINANCE_SYMBOLS = valid_symbols
    expected_symbols = sorted(
        s for s in fixed_universe_df["market_symbol"] if s in valid_symbols
    )
    log.info(f"\nSoll-Ticker: {len(expected_symbols)}")

    window_start = pd.Timestamp(OHLCV_START_DATE).date()
    window_end   = pd.Timestamp(OHLCV_END_DATE).date()

    files = sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet"))

    # ── First-Seen-Datum pro Ticker über alle Dateien hinweg ──
    log.info("\nErmittle First-Seen-Datum pro Ticker …")
    first_seen: dict[str, datetime.date] = {}
    for file in files:
        df_fs = pd.read_parquet(file, columns=["timestamp", "market_symbol"])
        df_fs = df_fs[df_fs["market_symbol"].isin(expected_symbols)]
        df_fs["date"] = pd.to_datetime(df_fs["timestamp"]).dt.date
        mins = df_fs.groupby("market_symbol")["date"].min()
        for symbol, d in mins.items():
            symbol = str(symbol)
            if symbol not in first_seen or d < first_seen[symbol]:
                first_seen[symbol] = d

    never_seen = [s for s in expected_symbols if s not in first_seen]
    log.info(f"   → {len(first_seen)}/{len(expected_symbols)} Ticker haben mindestens 1 Kerze irgendwann im Datenbestand")
    if never_seen:
        log.warning(f"   ⚠️  Noch nie im Datenbestand aufgetaucht (0 Kerzen im gesamten Zeitraum): {never_seen}")

    already_announced: set = set()   # Ticker, deren "Neu gelistet"-Meldung schon ausgegeben wurde
    total_gaps = []                  # echte Lücken (Ticker war ab dem Tag NACH first_seen bereits gelistet)

    for file in files:
        year, month = (int(x) for x in file.stem.removeprefix("ohlcv_").split("-"))
        month_first = pd.Timestamp(year=year, month=month, day=1).date()
        month_last  = pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)
        month_last  = month_last.date()
        expected_dates = pd.date_range(
            max(month_first, window_start), min(month_last, window_end), freq="D"
        ).date

        df = pd.read_parquet(file, columns=["timestamp", "market_symbol"])
        df["date"] = pd.to_datetime(df["timestamp"]).dt.date
        df = df[df["market_symbol"].isin(expected_symbols)]

        candle_count = df.groupby(["date", "market_symbol"]).size()
        dates_with_data = set(candle_count.index.get_level_values("date"))

        log.info(f"\n📄 {file.name}")
        log.info(f"   Soll-Tage   : {len(expected_dates)}")
        log.info(f"   Soll-Ticker : {len(expected_symbols)}")
        print()
        log.info(f"   {'Datum':<14} {'Vollständig':>12} {'Teilweise':>10} {'Fehlend/0':>10}")
        log.info(f"   {'-'*14} {'-'*12} {'-'*10} {'-'*10}")

        for date in expected_dates:
            if date in dates_with_data:
                counts = candle_count.xs(date, level="date").reindex(expected_symbols, fill_value=0)
            else:
                counts = pd.Series(0, index=expected_symbols)

            # Nur Ticker berücksichtigen, die an diesem Tag bereits gelistet sein sollten —
            # der First-Seen-Tag selbst zählt nicht als Lücke, da er strukturell nie volle
            # 288 Kerzen haben kann (Listing passiert meist mitten am Tag, nicht um 00:00 UTC)
            listed_symbols = [s for s in expected_symbols if s in first_seen and first_seen[s] < date]
            listed_counts  = counts.reindex(listed_symbols, fill_value=0)

            n_complete = (listed_counts >= 274).sum()
            n_partial  = ((listed_counts >= 200) & (listed_counts < 274)).sum()
            n_missing  = (listed_counts < 200).sum()

            gap_symbols = []
            for symbol, n in listed_counts.items():
                if n < 274:
                    gap_symbols.append(symbol)
                    total_gaps.append((file.name, date, symbol, int(n)))

            new_listings = [
                s for s in expected_symbols
                if first_seen.get(s) == date and s not in already_announced
            ]
            already_announced.update(new_listings)

            status = "✅" if n_missing == 0 and n_partial == 0 else "⚠️ " if n_missing == 0 else "❌"
            log.info(f"   {str(date):<14} {n_complete:>12} {n_partial:>10} {n_missing:>10}  {status}")
            if gap_symbols:
                log.info(f"      → echte Lücke:     {gap_symbols}")
            if new_listings:
                log.info(f"      🆕 Neu gelistet:   {new_listings}")

    log.info("\n" + "=" * 70)
    log.info(f"ECHTE LÜCKEN (Tag, Ticker) mit < 274 Kerzen, ab dem Tag nach First-Seen: {len(total_gaps)}")
    log.info("=" * 70)
    for fname, date, symbol, n in total_gaps:
        log.warning(f"   {fname}  {date}  {symbol:<14} {n:>3} Kerzen")

ohlcvCheck()

line: 7     INFO      ======================================================================
line: 8     INFO      OHLCV DATEI-ÜBERSICHT
line: 9     INFO      ======================================================================
line: 81    INFO      ────────────────────────────────────────────────────────────
line: 82    INFO       Fixe Asset-Universe laden
line: 83    INFO      ────────────────────────────────────────────────────────────
line: 85    INFO        → 57 Assets geladen
line: 89    INFO        → 47 Assets mit aufgelöstem Binance-Symbol
line: 93    WARNING     ⚠️  10 Assets ohne Binance-Symbol — werden übersprungen: ['CC', 'CRO', 'HYPE', 'KAS', 'LIT', 'MNT', 'STABLE', 'VVV', 'XDC', 'FLR']
line: 19    INFO      
Soll-Ticker: 47
line: 27    INFO      
Ermittle First-Seen-Datum pro Ticker …
line: 40    INFO         → 47/47 Ticker haben mindestens 1 Kerze irgendwann im Datenbestand
line: 63    INFO      
📄 ohlcv_2024-07.parquet
line: 64    INFO         Soll-Tage   : 9
line: 65

line: 64    INFO         Soll-Tage   : 31
line: 65    INFO         Soll-Ticker : 47
line: 67    INFO         Datum           Vollständig  Teilweise  Fehlend/0
line: 68    INFO         -------------- ------------ ---------- ----------
line: 99    INFO         2024-08-01               38          0          0  ✅
line: 99    INFO         2024-08-02               38          0          0  ✅
line: 99    INFO         2024-08-03               38          0          0  ✅
line: 99    INFO         2024-08-04               38          0          0  ✅
line: 99    INFO         2024-08-05               38          0          0  ✅
line: 99    INFO         2024-08-06               38          0          0  ✅
line: 99    INFO         2024-08-07               38          0          0  ✅
line: 99    INFO         2024-08-08               38          0          0  ✅
line: 99    INFO         2024-08-09               38          0          0  ✅
line: 99    INFO         2024-08-10               38          0 

line: 63    INFO      
📄 ohlcv_2024-09.parquet
line: 64    INFO         Soll-Tage   : 30
line: 65    INFO         Soll-Ticker : 47
line: 67    INFO         Datum           Vollständig  Teilweise  Fehlend/0
line: 68    INFO         -------------- ------------ ---------- ----------
line: 99    INFO         2024-09-01               38          0          0  ✅
line: 99    INFO         2024-09-02               38          0          0  ✅
line: 99    INFO         2024-09-03               38          0          0  ✅
line: 99    INFO         2024-09-04               38          0          0  ✅
line: 99    INFO         2024-09-05               38          0          0  ✅
line: 99    INFO         2024-09-06               38          0          0  ✅
line: 99    INFO         2024-09-07               38          0          0  ✅
line: 99    INFO         2024-09-08               38          0          0  ✅
line: 99    INFO         2024-09-09               38          0          0  ✅
line: 99    INFO 

line: 63    INFO      
📄 ohlcv_2024-10.parquet
line: 64    INFO         Soll-Tage   : 31
line: 65    INFO         Soll-Ticker : 47
line: 67    INFO         Datum           Vollständig  Teilweise  Fehlend/0
line: 68    INFO         -------------- ------------ ---------- ----------
line: 99    INFO         2024-10-01               39          0          0  ✅
line: 99    INFO         2024-10-02               39          0          0  ✅
line: 99    INFO         2024-10-03               39          0          0  ✅
line: 99    INFO         2024-10-04               39          0          0  ✅
line: 99    INFO         2024-10-05               39          0          0  ✅
line: 99    INFO         2024-10-06               39          0          0  ✅
line: 99    INFO         2024-10-07               39          0          0  ✅
line: 99    INFO         2024-10-08               39          0          0  ✅
line: 99    INFO         2024-10-09               39          0          0  ✅
line: 99    INFO 

line: 63    INFO      
📄 ohlcv_2024-11.parquet
line: 64    INFO         Soll-Tage   : 30
line: 65    INFO         Soll-Ticker : 47
line: 67    INFO         Datum           Vollständig  Teilweise  Fehlend/0
line: 68    INFO         -------------- ------------ ---------- ----------
line: 99    INFO         2024-11-01               39          0          0  ✅
line: 99    INFO         2024-11-02               39          0          0  ✅
line: 99    INFO         2024-11-03               39          0          0  ✅
line: 99    INFO         2024-11-04               39          0          0  ✅
line: 99    INFO         2024-11-05               39          0          0  ✅
line: 99    INFO         2024-11-06               39          0          0  ✅
line: 99    INFO         2024-11-07               39          0          0  ✅
line: 99    INFO         2024-11-08               39          0          0  ✅
line: 99    INFO         2024-11-09               39          0          0  ✅
line: 99    INFO 

...

line: 63    INFO      
📄 ohlcv_2026-03.parquet
line: 64    INFO         Soll-Tage   : 31
line: 65    INFO         Soll-Ticker : 47
line: 67    INFO         Datum           Vollständig  Teilweise  Fehlend/0
line: 68    INFO         -------------- ------------ ---------- ----------
line: 99    INFO         2026-03-01               45          0          0  ✅
line: 99    INFO         2026-03-02               45          0          0  ✅
line: 99    INFO         2026-03-03               45          0          0  ✅
line: 99    INFO         2026-03-04               45          0          0  ✅
line: 99    INFO         2026-03-05               45          0          0  ✅
line: 99    INFO         2026-03-06               45          0          0  ✅
line: 99    INFO         2026-03-07               45          0          0  ✅
line: 99    INFO         2026-03-08               45          0          0  ✅
line: 99    INFO         2026-03-09               45          0          0  ✅
line: 99    INFO 

line: 63    INFO      
📄 ohlcv_2026-04.parquet
line: 64    INFO         Soll-Tage   : 30
line: 65    INFO         Soll-Ticker : 47
line: 67    INFO         Datum           Vollständig  Teilweise  Fehlend/0
line: 68    INFO         -------------- ------------ ---------- ----------
line: 99    INFO         2026-04-01               46          0          0  ✅
line: 99    INFO         2026-04-02               46          0          0  ✅
line: 99    INFO         2026-04-03               46          0          0  ✅
line: 99    INFO         2026-04-04               46          0          0  ✅
line: 99    INFO         2026-04-05               46          0          0  ✅
line: 99    INFO         2026-04-06               46          0          0  ✅
line: 99    INFO         2026-04-07               46          0          0  ✅
line: 99    INFO         2026-04-08               46          0          0  ✅
line: 99    INFO         2026-04-09               46          0          0  ✅
line: 99    INFO 

line: 63    INFO      
📄 ohlcv_2026-05.parquet
line: 64    INFO         Soll-Tage   : 31
line: 65    INFO         Soll-Ticker : 47
line: 67    INFO         Datum           Vollständig  Teilweise  Fehlend/0
line: 68    INFO         -------------- ------------ ---------- ----------
line: 99    INFO         2026-05-01               46          0          0  ✅
line: 99    INFO         2026-05-02               46          0          0  ✅
line: 99    INFO         2026-05-03               46          0          0  ✅
line: 99    INFO         2026-05-04               46          0          0  ✅
line: 99    INFO         2026-05-05               46          0          0  ✅
line: 99    INFO         2026-05-06               46          0          0  ✅
line: 99    INFO         2026-05-07               46          0          0  ✅
line: 99    INFO         2026-05-08               46          0          0  ✅
line: 99    INFO         2026-05-09               46          0          0  ✅
line: 99    INFO 

line: 63    INFO      
📄 ohlcv_2026-06.parquet
line: 64    INFO         Soll-Tage   : 30
line: 65    INFO         Soll-Ticker : 47
line: 67    INFO         Datum           Vollständig  Teilweise  Fehlend/0
line: 68    INFO         -------------- ------------ ---------- ----------
line: 99    INFO         2026-06-01               46          0          0  ✅
line: 99    INFO         2026-06-02               46          0          0  ✅
line: 99    INFO         2026-06-03               46          0          0  ✅
line: 99    INFO         2026-06-04               46          0          0  ✅
line: 99    INFO         2026-06-05               46          0          0  ✅
line: 99    INFO         2026-06-06               46          0          0  ✅
line: 99    INFO         2026-06-07               46          0          0  ✅
line: 99    INFO         2026-06-08               46          0          0  ✅
line: 99    INFO         2026-06-09               46          0          0  ✅
line: 99    INFO 

line: 63    INFO      
📄 ohlcv_2026-07.parquet
line: 64    INFO         Soll-Tage   : 22
line: 65    INFO         Soll-Ticker : 47
line: 67    INFO         Datum           Vollständig  Teilweise  Fehlend/0
line: 68    INFO         -------------- ------------ ---------- ----------
line: 99    INFO         2026-07-01               46          0          0  ✅
line: 99    INFO         2026-07-02               46          0          0  ✅
line: 103   INFO            🆕 Neu gelistet:   ['GRAMUSDT']
line: 99    INFO         2026-07-03               47          0          0  ✅
line: 99    INFO         2026-07-04               47          0          0  ✅
line: 99    INFO         2026-07-05               47          0          0  ✅
line: 99    INFO         2026-07-06               47          0          0  ✅
line: 99    INFO         2026-07-07               47          0          0  ✅
line: 99    INFO         2026-07-08               47          0          0  ✅
line: 99    INFO         2026-07-09 

## 6 · Eine grosse Tabelle über den ganzen Zeitraum

Die Monats-Partitionierung (`ohlcv_YYYY-MM.parquet`) ist nur ein Speicher-Detail.
`load_full_ohlcv_table()` lädt alle Monatsdateien, prüft, dass sie identische
Spalten/Dtypes haben (deckt Schema-Drift sofort auf), und liefert damit die
eine logische Tabelle für alle Assets über den ganzen Zeitraum.

In [6]:
def load_full_ohlcv_table() -> pd.DataFrame:
    files = sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet"))
    if not files:
        raise FileNotFoundError(f"No OHLCV files found in {ASSET_OHLCV_5M_DIR}")

    frames = [pd.read_parquet(f) for f in files]

    reference_schema = frames[0].dtypes
    for f, frame in zip(files, frames):
        if not frame.dtypes.equals(reference_schema):
            raise ValueError(f"Schema drift detected in {f.name}: {frame.dtypes.to_dict()} != {reference_schema.to_dict()}")

    full_df = pd.concat(frames, ignore_index=True).sort_values(["market_symbol", "timestamp"]).reset_index(drop=True)
    return full_df


full_ohlcv_df = load_full_ohlcv_table()

print(f"Total rows:        {len(full_ohlcv_df):,}")
print(f"Distinct symbols:  {full_ohlcv_df['market_symbol'].nunique()}")
print(f"Date range:        {full_ohlcv_df['timestamp'].min()}  →  {full_ohlcv_df['timestamp'].max()}")
print(f"Columns:           {full_ohlcv_df.columns.tolist()}")

Total rows:        8,798,178
Distinct symbols:  47
Date range:        2024-07-23 00:00:00+00:00  →  2026-07-22 23:55:00+00:00
Columns:           ['timestamp', 'market_symbol', 'open', 'high', 'low', 'close', 'volume_base', 'volume_quote', 'trade_count']


## 7 · Zwischenergebnis-Inspektion

Zeigt alle Zwischenstufen der Datenpipeline im Detail an  nur zur Analyse und zum Verständnis des Datenflusses. Diese Zelle verändert keine Daten und muss nicht bei jedem Lauf ausgeführt werden.

**Dargestellte Stufen:**

| Stufe | Inhalt |
|---|---|
| `df_assets` | Alle Coins aus `assets.parquet` mit Metadaten |
| `fixed_universe_df` | Fixe Ticker-Liste (nur Assets mit aufgelöstem `binance_symbol`) |
| `raw` | Rohe Binance API-Antwort (Liste von 12-elementigen Arrays) |
| `all_rows` | Umgewandelte Dicts mit sprechenden Spaltennamen |
| `df` | Pandas DataFrame bereit zum Speichern |
| Parquet-Dateien | Finales Ergebnis mit Shape, Ticker- und Tagesanzahl |

In [7]:
# =============================================================
# ZWISCHENERGEBNIS-INSPEKTION — alle Stufen aus dem Diagramm
# =============================================================

SEP = "=" * 65

# ── 1. df_assets ─────────────────────────────────────────────
print(f"\n{SEP}")
print("1) df_assets   (aus assets.parquet)")
print(SEP)
df_assets = pd.read_parquet(ASSETS_PATH)
print(f"  Typ    : {type(df_assets)}")
print(f"  Shape  : {df_assets.shape}  →  {df_assets.shape[0]} Assets, {df_assets.shape[1]} Spalten")
print(f"  Spalten: {df_assets.columns.tolist()}")
print(f"  Dtypes :\n{df_assets.dtypes.to_string()}")
print("\n  Erste 3 Zeilen:")
display(df_assets.head(3))

# ── 2. fixed_universe_df ─────────────────────────────────────
print(f"\n{SEP}")
print("2) fixed_universe_df   (nur Assets mit aufgelöstem binance_symbol)")
print(SEP)
_fixed = load_fixed_asset_universe()
print(f"  Typ    : {type(_fixed)}")
print(f"  Shape  : {_fixed.shape}")
print(f"  Spalten: {_fixed.columns.tolist()}")
display(_fixed.head(5))

# ── 3. raw   (ein echter Binance-API Aufruf) ─────────────────
print(f"\n{SEP}")
print("3) raw   (Binance API Antwort — JSON Liste)")
print(SEP)
_example_symbol = _fixed["market_symbol"].iloc[0]
_raw = public_get("/api/v3/klines", params={
    "symbol":    _example_symbol,
    "interval":  "5m",
    "startTime": int(pd.Timestamp(OHLCV_START_DATE, tz="UTC").timestamp() * 1000),
    "endTime":   int(pd.Timestamp(f"{OHLCV_START_DATE} 00:30", tz="UTC").timestamp() * 1000),
    "limit":     6,
})
print(f"  Symbol          : {_example_symbol}")
print(f"  Typ             : {type(_raw)}")
print(f"  Anzahl Kerzen   : {len(_raw)}")
print(f"  Elemente pro Kerze: {len(_raw[0])}")
print("\n  Erste Kerze (rohe Werte):")
_labels = ["open_time","open","high","low","close","volume","close_time",
           "quote_volume","trades","taker_buy_base","taker_buy_quote","ignore"]
for label, val in zip(_labels, _raw[0]):
    print(f"    [{_labels.index(label)}] {label:<20} = {val}")

# ── 4. all_rows   (nach Umwandlung in Dicts) ─────────────────
print(f"\n{SEP}")
print("4) all_rows   (Liste von Dictionaries)")
print(SEP)
_all_rows = []
for _row in _raw:
    _all_rows.append({
        "timestamp":     pd.to_datetime(int(_row[0]), unit="ms", utc=True),
        "market_symbol": _example_symbol,
        "open":          float(_row[1]),
        "high":          float(_row[2]),
        "low":           float(_row[3]),
        "close":         float(_row[4]),
        "volume_base":   float(_row[5]),
        "volume_quote":  float(_row[7]),
        "trade_count":   int(_row[8]),
    })
print(f"  Typ             : {type(_all_rows)}")
print(f"  Anzahl Einträge : {len(_all_rows)}")
print(f"  Keys eines Eintrags: {list(_all_rows[0].keys())}")
print("\n  Erster Eintrag:")
for k, v in _all_rows[0].items():
    print(f"    {k:<20} = {v}  ({type(v).__name__})")

# ── 5. df = pd.DataFrame(all_rows) ──────────────────────────
print(f"\n{SEP}")
print("5) df = pd.DataFrame(all_rows)   (Tages-DataFrame)")
print(SEP)
_df_day = pd.DataFrame(_all_rows)
print(f"  Typ    : {type(_df_day)}")
print(f"  Shape  : {_df_day.shape}  →  {_df_day.shape[0]} Kerzen, {_df_day.shape[1]} Spalten")
print(f"  Dtypes :\n{_df_day.dtypes.to_string()}")
print("\n  Alle Zeilen:")
display(_df_day)

# ── 6. Finale Parquet-Dateien ────────────────────────────────
print(f"\n{SEP}")
print("6) parquet files pro Monat   (Endresultat)")
print(SEP)
_ohlcv_files = sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet"))
print(f"  Anzahl Dateien: {len(_ohlcv_files)}")
for _f in _ohlcv_files:
    _df_f = pd.read_parquet(_f)
    print(f"\n  📄 {_f.name}")
    print(f"     Shape  : {_df_f.shape}  →  {_df_f.shape[0]:,} Kerzen, {_df_f.shape[1]} Spalten")
    print(f"     Ticker : {_df_f['market_symbol'].nunique()}")
    print(f"     Tage   : {pd.to_datetime(_df_f['timestamp']).dt.date.nunique()}")
    print(f"     Von    : {_df_f['timestamp'].min()}")
    print(f"     Bis    : {_df_f['timestamp'].max()}")

print(f"\n{SEP}")
print("INSPEKTION ABGESCHLOSSEN")
print(SEP)



1) df_assets   (aus assets.parquet)
  Typ    : <class 'pandas.core.frame.DataFrame'>
  Shape  : (57, 9)  →  57 Assets, 9 Spalten
  Spalten: ['asset_id', 'coingecko_id', 'symbol', 'name', 'circulating_supply', 'total_supply', 'max_supply', 'top_exchange_count', 'binance_symbol']
  Dtypes :
asset_id                int64
coingecko_id           object
symbol                 object
name                   object
circulating_supply    float64
total_supply          float64
max_supply            float64
top_exchange_count      int64
binance_symbol         object

  Erste 3 Zeilen:


,asset_id,coingecko_id,symbol,name,circulating_supply,total_supply,max_supply,top_exchange_count,binance_symbol
0,1,aave,AAVE,Aave,1.541754e+07,1.600000e+07,1.600000e+07,19,AAVEUSDT
1,4,algorand,ALGO,Algorand,8.969018e+09,8.969018e+09,1.000000e+10,19,ALGOUSDT
2,5,aptos,APT,Aptos,8.450336e+08,1.206093e+09,2.100000e+09,18,APTUSDT


line: 81    INFO      ────────────────────────────────────────────────────────────
line: 82    INFO       Fixe Asset-Universe laden
line: 83    INFO      ────────────────────────────────────────────────────────────
line: 85    INFO        → 57 Assets geladen
line: 89    INFO        → 47 Assets mit aufgelöstem Binance-Symbol
line: 93    WARNING     ⚠️  10 Assets ohne Binance-Symbol — werden übersprungen: ['CC', 'CRO', 'HYPE', 'KAS', 'LIT', 'MNT', 'STABLE', 'VVV', 'XDC', 'FLR']



2) fixed_universe_df   (nur Assets mit aufgelöstem binance_symbol)
  Typ    : <class 'pandas.core.frame.DataFrame'>
  Shape  : (47, 5)
  Spalten: ['asset_id', 'coingecko_id', 'symbol', 'name', 'market_symbol']


,asset_id,coingecko_id,symbol,name,market_symbol
0,1,aave,AAVE,Aave,AAVEUSDT
1,4,algorand,ALGO,Algorand,ALGOUSDT
2,5,aptos,APT,Aptos,APTUSDT
3,6,arbitrum,ARB,Arbitrum,ARBUSDT
4,7,aster-2,ASTER,Aster,ASTERUSDT



3) raw   (Binance API Antwort — JSON Liste)
  Symbol          : AAVEUSDT
  Typ             : <class 'list'>
  Anzahl Kerzen   : 6
  Elemente pro Kerze: 12

  Erste Kerze (rohe Werte):
    [0] open_time            = 1721692800000
    [1] open                 = 94.16000000
    [2] high                 = 94.35000000
    [3] low                  = 94.06000000
    [4] close                = 94.21000000
    [5] volume               = 135.79400000
    [6] close_time           = 1721693099999
    [7] quote_volume         = 12784.39570000
    [8] trades               = 104
    [9] taker_buy_base       = 58.07200000
    [10] taker_buy_quote      = 5467.91279000
    [11] ignore               = 0

4) all_rows   (Liste von Dictionaries)
  Typ             : <class 'list'>
  Anzahl Einträge : 6
  Keys eines Eintrags: ['timestamp', 'market_symbol', 'open', 'high', 'low', 'close', 'volume_base', 'volume_quote', 'trade_count']

  Erster Eintrag:
    timestamp            = 2024-07-23 00:00:00+00:00  (Ti

,timestamp,market_symbol,open,high,low,close,volume_base,volume_quote,trade_count
0,2024-07-23 00:00:00+00:00,AAVEUSDT,94.16,94.35,94.06,94.21,135.794,12784.39570,104
1,2024-07-23 00:05:00+00:00,AAVEUSDT,94.20,94.42,94.20,94.41,48.344,4559.15542,69
2,2024-07-23 00:10:00+00:00,AAVEUSDT,94.43,94.61,94.26,94.42,602.009,56869.03603,225
3,2024-07-23 00:15:00+00:00,AAVEUSDT,94.42,94.49,94.31,94.34,94.698,8936.03250,90
4,2024-07-23 00:20:00+00:00,AAVEUSDT,94.36,94.44,94.17,94.17,172.779,16307.84613,100
5,2024-07-23 00:25:00+00:00,AAVEUSDT,94.17,94.23,94.09,94.11,42.546,4005.79429,54



6) parquet files pro Monat   (Endresultat)
  Anzahl Dateien: 25

  📄 ohlcv_2024-07.parquet
     Shape  : (97536, 9)  →  97,536 Kerzen, 9 Spalten
     Ticker : 38
     Tage   : 9
     Von    : 2024-07-23 00:00:00+00:00
     Bis    : 2024-07-31 23:55:00+00:00

  📄 ohlcv_2024-08.parquet
     Shape  : (339264, 9)  →  339,264 Kerzen, 9 Spalten
     Ticker : 38
     Tage   : 31
     Von    : 2024-08-01 00:00:00+00:00
     Bis    : 2024-08-31 23:55:00+00:00

  📄 ohlcv_2024-09.parquet
     Shape  : (333384, 9)  →  333,384 Kerzen, 9 Spalten
     Ticker : 39
     Tage   : 30
     Von    : 2024-09-01 00:00:00+00:00
     Bis    : 2024-09-30 23:55:00+00:00

  📄 ohlcv_2024-10.parquet
     Shape  : (348192, 9)  →  348,192 Kerzen, 9 Spalten
     Ticker : 39
     Tage   : 31
     Von    : 2024-10-01 00:00:00+00:00
     Bis    : 2024-10-31 23:55:00+00:00

  📄 ohlcv_2024-11.parquet
     Shape  : (336960, 9)  →  336,960 Kerzen, 9 Spalten
     Ticker : 39
     Tage   : 30
     Von    : 2024-11-01 00:00:00